# Groundwater Pumping and Baseflow in the Republican River Basin

How much does irrigation pumping reduce baseflow once weather, location and season are accounted for? The goal is to help water managers in the basin plan how much pumping the river can handle.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import geopandas as gpd
import contextily as cx

## Loading the data

In [ ]:
df = pd.read_csv('data/RRCA_baseflow.csv')

# Date is a matlab day number
df['DateTime'] = pd.to_datetime(df['Date'] - 719529, unit='D')
df['DateTime'].head(10)

Jan 29 shows up right after Jan 1 so some of these are actually the next month. Rounding to the nearest month fixes it.

In [ ]:
df['DateTime'] = (df['DateTime'] + pd.Timedelta(days=15)).dt.to_period('M').dt.to_timestamp()
df['Year'] = df['DateTime'].dt.year
df['Month'] = df['DateTime'].dt.month

# pumping is stored as negative numbers
df['Pumping'] = df['Irrigation_pumping'] * -1

display(df.head(5))
df.describe()

Where/When is this?
Dates: 10/2/1939 - 9/1/2000

Republican River Basin - a 15.9-million-acre watershed spanning northeastern Colorado, southwestern Nebraska, and northwestern Kansas

In [ ]:
print(f'min {df['DateTime'].min()} max {df['DateTime'].max()}')
print(f'Number of segments: {df['Segment_id'].nunique()}')

date_ranges = (
    df.groupby('Segment_id')['DateTime']
      .agg(first='min', last='max')
      .sort_values('first')
)

fig, ax = plt.subplots(figsize=(12, 8))

for segment_id, row in date_ranges.iterrows():
    ax.plot([row['first'], row['last']], [segment_id, segment_id],
            color='steelblue', linewidth=2)
    ax.scatter([row['first'], row['last']], [segment_id, segment_id],
               color=['green', 'red'], zorder=3)

ax.set_xlabel('Date')
ax.set_ylabel('Segment ID')
ax.set_title('First and Last Date Present by Segment')
ax.grid(axis='x', alpha=0.3)
fig.autofmt_xdate()
plt.show()

## Segments

There are duplicate segment/date rows, checking why

In [ ]:
df.groupby('Segment_id')[['x', 'y']].nunique().sort_values('x', ascending=False).head()

In [ ]:
df[df['Segment_id'] == 53].groupby(['x', 'y'])['Observed'].agg(['count', 'mean'])

Some segments have 2 gauges at different spots with really different flows, so I'm making a station id from x/y instead of using Segment_id.

In [ ]:
df['Station'] = df.groupby(['x', 'y']).ngroup()
df['Station'].nunique()

In [ ]:
df.groupby('Station').agg({'Segment_id': 'first', 'Year': ['min', 'max'], 'Observed': 'count'})

### Screen for Stations With Bad Data
Check if any stations have their precip, evapotranspiration, or pumping set to 0 for most of their rows.

In [ ]:
df_sorted = df.sort_values('DateTime')

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
for station, group in df_sorted.groupby('Station'):
    axes[0].plot(group['DateTime'], group['Precipitation'], label=station, linewidth=1, alpha=0.6)
    axes[1].plot(group['DateTime'], group['Evapotranspiration'], label=station, linewidth=1, alpha=0.6)
    axes[2].plot(group['DateTime'], group['Pumping'], label=station, linewidth=1, alpha=0.6)
    axes[3].plot(group['DateTime'], group['Observed'], label=station, linewidth=1, alpha=0.6)

axes[3].set_xlabel('DateTime')
axes[0].set_ylabel('Precip')
axes[1].set_ylabel('Evapotranspiration')
axes[2].set_ylabel('Pumping')
axes[3].set_ylabel('Baseflow')

axes[0].legend(title='Station', ncol=4, fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

Grouping by station instead of segment gets rid of the solid filled areas (those were two gauges sharing a segment id). Some stations sit at 0 the whole time and the long diagonal lines are gaps of missing months.

## Zeros

In [ ]:
(df['Precipitation'] == 0).mean()

In [ ]:
df.groupby('Station')['Precipitation'].apply(lambda x: (x == 0).mean()).sort_values(ascending=False).head(12)

In [ ]:
df[df['Station'] == 0][['Precipitation', 'Evapotranspiration', 'Pumping']].describe()

In [ ]:
df[(df['Station'] == 38) & (df['Precipitation'] == 0)]['Year'].unique()

9 stations have 0 for precipitation, ET, and pumping the entire time so that's missing data not real zeros. Station 38 loses its precipitation after 1994.

Also precipitation looks like the same number every month of a year:

In [ ]:
df[df['Station'] == 15][['DateTime', 'Precipitation', 'Evapotranspiration']].head(15)

So precipitation is yearly. A whole year with 0 rain isn't realistic so I'm dropping rows where precipitation is 0.

In [ ]:
df = df[df['Precipitation'] > 0]
print(len(df), df['Station'].nunique())

## Map

In [ ]:
stations = df.drop_duplicates('Station')
gdf = gpd.GeoDataFrame(stations, geometry=gpd.points_from_xy(stations.x * 0.3048, stations.y * 0.3048),
                       crs='EPSG:26914').to_crs(epsg=3857)   # UTM 14 (ft -> m) -> web mercator
ax = gdf.plot(figsize=(12, 6), color='red', markersize=30, edgecolor='k')
cx.add_basemap(ax, source=cx.providers.Esri.WorldImagery)
ax.set_xlabel('Easting [metre]')
ax.set_ylabel('Northing [metre]')
plt.show()

## Baseflow

In [ ]:
sns.histplot(df['Observed'], bins=100)
plt.show()

In [ ]:
df[df['Observed'] < 0]['Station'].value_counts()

Super skewed, most is under 25 but a few go over 700. Negatives are almost all station 9, which is probably a losing stream (river leaking into the ground). Keeping them since they're real.

In [ ]:
station_means = df.groupby('Station')[['x', 'y', 'Observed', 'Pumping']].mean()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=station_means, x='x', y='y', hue='Observed', size='Observed')
plt.title('avg baseflow')
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=station_means, x='x', y='y', hue='Pumping', size='Pumping')
plt.title('avg pumping')
plt.show()

The two biggest baseflow stations are on the east end (downstream, river flows into Kansas). Most pumping is on the west side.

## Over time

In [ ]:
yearly = df.groupby('Year')[['Observed', 'Pumping', 'Precipitation', 'Evapotranspiration']].mean()
yearly.plot(subplots=True, figsize=(10, 9))
plt.show()

Baseflow drops a ton around 1950 but I think that's just some big stations stop getting recorded then. Pumping starts in the mid 50s and goes up until about 1980. Rain and ET don't really trend.

In [ ]:
sns.boxplot(data=df, x='Month', y='Observed', showfliers=False)
plt.show()

Baseflow is lowest Jul-Sep, which is when pumping happens, so season has to be in the model or pumping gets blamed for what the season is doing.

## Looking at single stations

In [ ]:
station_means.sort_values('Pumping', ascending=False).head()

Picking station 7 (lots of pumping and a long record), station 4 (not much pumping), and station 9 (the losing stream).

In [ ]:
for s in [7, 4, 9]:
    d = df[df['Station'] == s]
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(d['DateTime'], d['Observed'])
    ax.set_ylabel('baseflow')
    ax2 = ax.twinx()
    ax2.plot(d['DateTime'], d['Pumping'], color='red')
    ax2.set_ylabel('pumping', color='red')
    plt.title('station ' + str(s))
    plt.show()

Pumping only happens in summer, so it's gonna be tied up with month. Station 7 slowly drops after pumping starts in the 50s. Station 9 starts going negative in the summers once pumping picks up.

In [ ]:
corrs = []
for s in df['Station'].unique():
    d = df[df['Station'] == s]
    if d['Pumping'].max() > 0:
        corrs.append([s, d['Observed'].corr(d['Pumping'])])

corrs = pd.DataFrame(corrs, columns=['Station', 'corr']).sort_values('corr')
corrs.plot(x='Station', y='corr', kind='bar', figsize=(12, 4), legend=False)
plt.title('baseflow vs pumping correlation for each station')
plt.show()

Inside basically every station more pumping = less baseflow. Only station 17 goes the other way.

## Baseflow vs everything

Pooling all the stations together hides that relationship:

In [ ]:
sns.scatterplot(data=df, x='Pumping', y='Observed', alpha=0.2, s=10)
plt.show()
sns.scatterplot(data=df, x='Precipitation', y='Observed', alpha=0.2, s=10)
plt.show()
sns.scatterplot(data=df, x='Evapotranspiration', y='Observed', alpha=0.2, s=10)
plt.show()

In [ ]:
sns.heatmap(df[['Observed', 'Pumping', 'Precipitation', 'Evapotranspiration', 'x', 'y']].corr(), annot=True, cmap='coolwarm')
plt.show()

Correlations over everything are pretty weak. Pumping and ET are correlated (0.46) since they're both high in summer.

# Regression

How much does irrigation pumping reduce baseflow after accounting for weather?

Train on before 1990, test on 1990-2000. One station only has data after 1995 so it gets dropped from test.

In [ ]:
train = df[df['Year'] < 1990]
test = df[df['Year'] >= 1990]
test = test[test['Station'].isin(train['Station'])]

print(len(train), len(test))

## Model 1: just pumping

In [ ]:
m1 = smf.ols('Observed ~ Pumping', data=train).fit()
m1.summary()

## Model 2: add rain and ET

In [ ]:
m2 = smf.ols('Observed ~ Pumping + Precipitation + Evapotranspiration', data=train).fit()
m2.summary()

## Model 3: add station

In [ ]:
m3 = smf.ols('Observed ~ Pumping + Precipitation + Evapotranspiration + C(Station)', data=train).fit()
print(m3.params[['Pumping', 'Precipitation', 'Evapotranspiration']])
print(m3.rsquared)

## Model 4: add month

In [ ]:
m4 = smf.ols('Observed ~ Pumping + Precipitation + Evapotranspiration + C(Station) + C(Month)', data=train).fit()
print(m4.params[['Pumping', 'Precipitation', 'Evapotranspiration']])
print(m4.rsquared)

## Autocorrelation

Durbin-Watson in the summaries is like 0.05 (should be around 2). Checking the residuals over time

In [ ]:
d = train[train['Station'] == 7]
plt.figure(figsize=(12, 3))
plt.plot(d['DateTime'], m4.resid[d.index])
plt.axhline(0, color='black')
plt.title('model 4 residuals, station 7')
plt.show()

In [ ]:
r = m4.resid
prev = r.groupby(train['Station']).shift()
r.corr(prev)

Residuals are 0.88 correlated with the month before so the normal p-values are way too confident. Using clustered standard errors by station to fix that.

## Comparing models

In [ ]:
def rmse(actual, pred):
    return np.sqrt(((actual - pred) ** 2).mean())

baseline = rmse(test['Observed'], train['Observed'].mean())
results = [{'Model': 'Baseline: always guess the average', 'Test RMSE': baseline}]

formulas = ['Observed ~ Pumping',
            'Observed ~ Pumping + Precipitation + Evapotranspiration',
            'Observed ~ Pumping + Precipitation + Evapotranspiration + C(Station)',
            'Observed ~ Pumping + Precipitation + Evapotranspiration + C(Station) + C(Month)']
names = ['1. Pumping only', '2. + rain and ET', '3. + station', '4. + month']

for name, f in zip(names, formulas):
    m = smf.ols(f, data=train).fit()
    mc = smf.ols(f, data=train).fit(cov_type='cluster', cov_kwds={'groups': train['Station']})
    results.append({'Model': name,
                    'Pumping effect': m.params['Pumping'],
                    'p': m.pvalues['Pumping'],
                    'Clustered p': mc.pvalues['Pumping'],
                    'R2': m.rsquared,
                    'Test RMSE': rmse(test['Observed'], m.predict(test))})

pd.DataFrame(results)[['Model', 'Pumping effect', 'p', 'Clustered p', 'R2', 'Test RMSE']].round(4)

Pumping is negative every time but it shrinks a lot once station and month are in. With clustered p-values it's only significant in models 1 and 2.

Model 4 basically kills it, I think because pumping only happens in summer so month is soaking up the effect. Station is what actually matters, r2 goes from 0.03 to 0.81 and the test error gets cut in half.

## Model 5: pumping over the last year

Groundwater pumping should take a while to affect the river, so trying the total pumped over the past 12 months instead of just that month.

In [ ]:
df = df.sort_values(['Station', 'DateTime'])
df['Pumping_12'] = df.groupby('Station')['Pumping'].transform(lambda x: x.rolling(12).sum())

train12 = df[df['Year'] < 1990].dropna()
test12 = df[df['Year'] >= 1990].dropna()
test12 = test12[test12['Station'].isin(train12['Station'])]

f5 = 'Observed ~ Pumping_12 + Precipitation + Evapotranspiration + C(Station) + C(Month)'
m5 = smf.ols(f5, data=train12).fit(cov_type='cluster', cov_kwds={'groups': train12['Station']})

print('coef:', m5.params['Pumping_12'])
print('clustered p:', m5.pvalues['Pumping_12'])
print('r2:', m5.rsquared)
print('test rmse:', rmse(test12['Observed'], m5.predict(test12)))

results.append({'Model': '5. Model 4 with past-12-month pumping',
                'Pumping effect': m5.params['Pumping_12'],
                'Clustered p': m5.pvalues['Pumping_12'],
                'R2': m5.rsquared,
                'Test RMSE': rmse(test12['Observed'], m5.predict(test12))})

-2.15 so each extra unit of pumping over a year is about 2 less baseflow. p = 0.087 so it's borderline. Test error is the lowest so far (10.0) but it's on a few less rows since the first 11 months of each station get dropped.

## Residuals

In [ ]:
plt.scatter(m4.fittedvalues, m4.resid, alpha=0.2, s=5)
plt.axhline(0, color='black')
plt.xlabel('fitted')
plt.ylabel('residual')
plt.show()

sns.histplot(m4.resid, bins=100)
plt.show()

The clump way out at ~400 is one huge station and its errors are massive. Errors get bigger as fitted goes up so there's heteroscedasticity.

## Model 6: log baseflow

Trying log since baseflow is so skewed. Can't log negatives so only rows with baseflow > 0.

In [ ]:
train_pos = train[train['Observed'] > 0]
test_pos = test[test['Observed'] > 0]

f6 = 'np.log(Observed) ~ Pumping + Precipitation + Evapotranspiration + C(Station) + C(Month)'
m6 = smf.ols(f6, data=train_pos).fit(cov_type='cluster', cov_kwds={'groups': train_pos['Station']})

print('coef:', m6.params['Pumping'])
print('% change per unit pumping:', (np.exp(m6.params['Pumping']) - 1) * 100)
print('clustered p:', m6.pvalues['Pumping'])
print('r2:', m6.rsquared)

In [ ]:
plt.scatter(m6.fittedvalues, m6.resid, alpha=0.2, s=5)
plt.axhline(0, color='black')
plt.xlabel('fitted')
plt.ylabel('residual')
plt.show()

In [ ]:
pred = np.exp(m6.predict(test_pos))
print('log model rmse:', rmse(test_pos['Observed'], pred))
print('model 4 rmse on same rows:', rmse(test_pos['Observed'], m4.predict(test_pos)))

results.append({'Model': '6. Model 4 on log baseflow',
                'Pumping effect': (np.exp(m6.params['Pumping']) - 1) * 100,
                'Clustered p': m6.pvalues['Pumping'],
                'R2': m6.rsquared,
                'Test RMSE': rmse(test_pos['Observed'], pred)})

Residuals look a lot more even with the log. Pumping comes out to about -11% baseflow per unit but it's not significant (p = 0.46). Test error is a little better than model 4.

## Multicollinearity

Pumping and ET were correlated so checking VIF

In [ ]:
X = train[['Pumping', 'Precipitation', 'Evapotranspiration']].copy()
X['const'] = 1
for i in range(3):
    print(X.columns[i], variance_inflation_factor(X.values, i))

All around 1.2 so it's not a problem.

## Table 1: all models

Model 6's pumping effect is % change in baseflow per unit pumping. Models 5 and 6 are tested on slightly different rows (model 5 drops the first 11 months of each station, model 6 drops baseflow <= 0), so their test error isn't a perfect apples-to-apples comparison.

In [ ]:
pd.DataFrame(results)[['Model', 'Pumping effect', 'p', 'Clustered p', 'R2', 'Test RMSE']].round(3)

## Test set

In [ ]:
test = test.copy()
test['Pred'] = m4.predict(test)

plt.scatter(test['Observed'], test['Pred'], alpha=0.3, s=8)
plt.plot([0, 120], [0, 120], color='black')
plt.xlabel('actual')
plt.ylabel('predicted')
plt.show()

In [ ]:
for s in [19, 7, 36]:
    d = test[test['Station'] == s]
    plt.figure(figsize=(12, 3))
    plt.plot(d['DateTime'], d['Observed'], label='actual')
    plt.plot(d['DateTime'], d['Pred'], label='predicted')
    plt.title('station ' + str(s))
    plt.legend()
    plt.show()

In [ ]:
for s in test['Station'].unique():
    d = test[test['Station'] == s]
    print(s, round(rmse(d['Observed'], d['Pred']), 2))

It gets the up and down pattern for some stations, but the month effect is the same for every station so steady ones like station 7 get way too big of swings. Station 8 is the worst.

## Takeaways and next steps

- Location matters far more than anything else: adding station takes R² from 0.03 to 0.81.
- Pumping is always negative, and it shows up best as total pumping over the past 12 months, but after clustering by station it's only borderline (p = 0.087).
- If we did this again we'd let each station have its own seasonal pattern (`C(Station):C(Month)`), try a separate model per station, and add last month's baseflow as an input to deal with the month-to-month correlation.